# Ex.6: Text-to-image and image-to-image with Stable Diffusion
**EECE.4860/5860**: ungraded practice for Lecture 12. **Needs a GPU** (Colab T4); about 3 minutes. Model: Stable Diffusion 1.5 (CreativeML OpenRAIL-M license; the safety checker stays on).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/acanets/genai-labs/blob/spring27/exercises/ex6_text_to_image.ipynb)

In [ ]:
import importlib.util
if importlib.util.find_spec("genai_utils") is None:
    get_ipython().run_line_magic("pip", 'install -q "git+https://github.com/acanets/genai-labs@spring27"')
import torch, matplotlib.pyplot as plt
from diffusers import AutoPipelineForText2Image, AutoPipelineForImage2Image
from genai_utils import setup_env, measure
device, dtype = setup_env()
SD = "stable-diffusion-v1-5/stable-diffusion-v1-5"
t2i = AutoPipelineForText2Image.from_pretrained(SD, torch_dtype=dtype).to(device)

## 1. Text-to-image: the same seed, different guidance scales

In [ ]:
prompt = "a lighthouse on a rocky coast at sunrise, oil painting"
imgs = []
with measure("3 images"):
    for g in (2.0, 7.5, 15.0):
        gen = torch.Generator("cpu").manual_seed(0)
        imgs.append(t2i(prompt, guidance_scale=g, num_inference_steps=25, generator=gen).images[0])
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, im, g in zip(ax, imgs, (2.0, 7.5, 15.0)):
    a.imshow(im); a.set_title(f"guidance {g}"); a.axis("off")
plt.show()

## 2. Image-to-image: how much should the new image depart from the old one?
`strength` sets how much noise is added to the input before denoising: 0 keeps it, 1 ignores it.

In [ ]:
i2i = AutoPipelineForImage2Image.from_pipe(t2i)
new_prompt = "a lighthouse on a rocky coast in a snowstorm at night, oil painting"
outs = []
for s in (0.3, 0.6, 0.9):
    gen = torch.Generator("cpu").manual_seed(0)
    outs.append(i2i(new_prompt, image=imgs[1], strength=s, guidance_scale=7.5, generator=gen).images[0])
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for a, im, t in zip(ax, [imgs[1]] + outs, ["input", "strength 0.3", "strength 0.6", "strength 0.9"]):
    a.imshow(im); a.set_title(t); a.axis("off")
plt.show()

## 3. Try it
* Change `num_inference_steps` to 5, 10, 50. Where do extra steps stop helping?
* Add a `negative_prompt="blurry, low quality"` and compare.